In [ ]:
import torch
gpu_id = 2
torch.cuda.set_device(gpu_id)
import cv2
import numpy as np
import sys
import os
from omegaconf import OmegaConf
import torchvision

sys.path.insert(0, '..')
sys.path.insert(0, os.path.join('..', 'third_party/core_stack'))

from tools.run_infinity import *
import warnings
warnings.filterwarnings("ignore") 

device = f'cuda:{gpu_id}'

model_path = '/home/yichen-xie/src/Infinity/checkpoints/Infinity-3ot255_7747edd2bbb_02000000_0/ar-gpt_model_latest.pth'
folder_name = os.path.dirname(model_path)
config_path = os.path.join(folder_name, "config.yaml")

dataset_name = 'nuplan'  # [nuplan, nuscenes, waymo]
save_format = 'gif'  # 'gif' or 'mp4'. MP4 is recommended for long videos

use_first_frame = 0  # Number of starting frames as the initial frame conditions (possible values: 0, 1, 2, 3)

num_frames = 10  # Number of frames to generate
slides = 1  # Step size of sliding window (don't change this)

assert os.path.exists(config_path), f"config does not exist at {config_path}"

config = OmegaConf.load(config_path)

text_encoder_ckpt = '../weights/flan-t5-xl'
if config.vae_type == 32:
    vae_path='../weights/infinity_vae_d32reg.pth'
    model_type = 'raynova_2b'
else:
    vae_path='../weights/infinity_vae_d16.pth'
    model_type = 'raynova_layer12'

num_views = 8

timesteps = 4
object_condition = config.object_condition
class_names = config.class_names
class_names = [
    'car', 'truck', 'construction_vehicle', 'bus', 'trailer', 'traffic_barrier',
    'motorcycle', 'bicycle', 'pedestrian', 'traffic_cone', 'vehicle', 'cyclist',
    'debris', 'traffic_light', 'unknown',
]
map_condition = getattr(config, 'map_condition', False)
map_names = getattr(config, 'map_names', [])
map_sample_points_num = getattr(config, 'map_sample_points_num', 30)
condition_rope = getattr(config, 'condition_rope', 0)
freeze_backbone = getattr(config, 'freeze_backbone', False)
view_embed_type = config.view_embed_type
use_temporal_condition = config.use_temporal_condition
attn_bias_type = config.attn_bias_type

num_slides = (num_frames - timesteps) // slides

pn=config.pn

args=argparse.Namespace(
    pn=pn,
    model_path=model_path,
    cfg_insertion_layer=0,
    vae_type=config.vae_type,
    vae_path=vae_path,
    add_lvl_embeding_only_first_block=config.add_lvl_embeding_only_first_block,
    add_view_embeding_only_first_block=config.add_view_embeding_only_first_block, 
    use_bit_label=config.use_bit_label,
    model_type=model_type,
    rope2d_each_sa_layer=config.rope2d_each_sa_layer,
    rope2d_normalized_by_hw=config.rope2d_normalized_by_hw,
    use_scale_schedule_embedding=0, # as in mv and the original
    sampling_per_bits=1,
    text_encoder_ckpt=text_encoder_ckpt,
    text_channels=2048,
    apply_spatial_patchify=config.apply_spatial_patchify,
    h_div_w_template=1,
    use_flex_attn=0,
    cache_dir='/dev/shm', # '/dev/shm'
    checkpoint_type=config.checkpoint_type,
    seed=0,
    bf16=1,
    save_file='tmp.jpg',
    enable_model_cache=True,  # True if using the trained checkpoints
    num_views=num_views,
    timesteps=timesteps,
    view_embed_type=view_embed_type,
    attn_bias_type=attn_bias_type,
    object_condition=object_condition,
    class_names=class_names,
    add_time_embeding_only_first_block=config.add_time_embeding_only_first_block,
    time_embed_type=config.time_embed_type,
    head_depth=config.dec,
    bbox_img_coord=config.bbox_img_coord,
    use_temporal_attn=use_temporal_condition,
    condition_rope=condition_rope,
    map_condition=map_condition,
    map_names=map_names,
    map_sample_points_num=map_sample_points_num,
    freeze_backbone=freeze_backbone,
)

/home/yichen-xie/miniconda3/envs/inf_stream/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
2026-04-28 16:39:35.090824: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

2026-04-28 16:39:35.896942: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/home/yichen-xie/miniconda3/envs/inf_stream/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
# load text encoder
text_tokenizer, text_encoder = load_tokenizer(t5_path=args.text_encoder_ckpt)
# load vae
vae = load_visual_tokenizer(args)

# load infinity
infinity = load_transformer_raynova(vae, args)
if hasattr(infinity, 'bbox_encoder'):
    infinity.bbox_encoder.prepare(text_tokenizer, text_encoder, class_names)

[Loading tokenizer and text encoder]


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

model_path: /home/yichen-xie/src/Infinity/checkpoints/Infinity-3ot255_7747edd2bbb_02000000_0/ar-gpt_model_latest.pth, slim_model_path: /home/yichen-xie/src/Infinity/checkpoints/Infinity-3ot255_7747edd2bbb_02000000_0/slim-gpt_model_latest.pth
local_model_path: /dev/shm/tmp/_home_yichen-xie_src_Infinity_checkpoints_Infinity-3ot255_7747edd2bbb_02000000_0_ar-gpt_model_latest.pth, local_slim_model_path: /dev/shm/tmp/_home_yichen-xie_src_Infinity_checkpoints_Infinity-3ot255_7747edd2bbb_02000000_0_slim-gpt_model_latest.pth
load checkpoint from /dev/shm/tmp/_home_yichen-xie_src_Infinity_checkpoints_Infinity-3ot255_7747edd2bbb_02000000_0_slim-gpt_model_latest.pth
[Loading InfinityStream]
self.codebook_dim: 32, self.add_lvl_embeding_only_first_block: 1,             self.use_bit_label: 1, self.rope2d_each_sa_layer: 1, self.rope2d_normalized_by_hw: 2
self.view_embed_type: local_plucker_ray_prope_6D_scale, self.add_view_embeding_only_first_block: 1
self.top_p: 0 self.top_k: 0
self.num_blocks_in_a_c

In [3]:
from scenarionet_tools.nuplan_data_wrapper import NuPlanDataWrapper, convert_bbox

from scenarionet_tools.map_utils import convert_points
from nuscenes_tools.nuscenes_utils.box3d_instance import LiDARInstance3DBoxes
from infinity.utils.vis_utils import draw_box_on_imgs, draw_map_points_on_imgs

os.environ['AWS_ACCESS_KEY_ID'] = 'cd0146b0fd5c24625a928b242d19f7e0dec18424'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'HN7mDT0pooo+3E40lyab8rrNfIied/33pCbpyrSEDuA='
os.environ['PREAUTH_URL'] = 'https://idskhu5vqvtl.objectstorage.us-phoenix-1.oci.customer-oci.com/p/ofkGTeRQaWyr0mNvkheVidOQYGEjr4OmEhEAi3EECl_UjuMeqtvu8mKr-k22ixWw/n/idskhu5vqvtl/b/research_datasets/o/remote_deps'

# os.environ['AWS_DEFAULT_REGION'] = 'us-phoenix-1'
# os.environ['AWS_ENDPOINT_URL'] = 'https://idskhu5vqvtl.compat.objectstorage.us-phoenix-1.oraclecloud.com'

os.environ['AWS_DEFAULT_REGION'] = 'us-chicago-1'
os.environ['AWS_ENDPOINT_URL'] = 'https://idskhu5vqvtl.compat.objectstorage.us-chicago-1.oraclecloud.com'



src_size = (1200, 2000)
cams = ['CAM_L2', 'CAM_L1', 'CAM_L0', 'CAM_F0', 'CAM_R0', 'CAM_R1', 'CAM_R2', 'CAM_B0']

mean=[0.5, 0.5, 0.5]
std=[0.5, 0.5, 0.5]
if pn == '0.25M':
    img_h, img_w = 384, 672
else:
    img_h, img_w = 192, 336

In [4]:
def generation(sample):
    cfg = 1  # classifier free guidance
    tau = 0.5
    # h_div_w = 1/1 # aspect ratio, height:width
    h_div_w = img_h / img_w # aspect ratio, height:width

    # seed = random.randint(0, 10000)
    seed = 0
    enable_positive_prompt=0

    h_div_w_template_ = h_div_w_templates[np.argmin(np.abs(h_div_w_templates-h_div_w))]
    scale_schedule = dynamic_resolution_h_w[h_div_w_template_][pn]['scales']
    scale_schedule = [(1, h, w) for (_, h, w) in scale_schedule]
    
    curr_to_first_lidar = torch.eye(4)
    curr_to_first_lidar_list = [curr_to_first_lidar]
    for item in sample[1:]:
        meta_data = item['img_metas']
        
        # meta_data['curr_to_prev_lidar_rt'] = torch.Tensor(
        #     [[0.5, -np.sqrt(3)/2, 0, 0], [np.sqrt(3)/2, 0.5, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]]
        # )
        
        # curr_to_first_lidar = curr_to_first_lidar @ meta_data['curr_to_prev_lidar_rt']
        curr_to_first_lidar_list.append(curr_to_first_lidar)

    temporal_gt_ls_Bl = {}
    video_imgs = []
    for slide_id in range(num_slides+1):
        print("Slide id:", slide_id, num_slides+1)

        camera_meta_data_list = []
        slide_start = slide_id * slides
        slide_end = slide_id * slides + timesteps
        first_to_seq_lidar = torch.inverse(sample[slide_start]['img_metas']['curr_to_first_lidar_rt'])

        for frame_id in range(slide_start, slide_end):
            item = sample[frame_id]
            meta_data = item['img_metas']
            batch_seq_ids = torch.zeros(1).int()  # No need to check sequence validation
            curr_to_prev_lidar = meta_data['curr_to_prev_lidar_rt']  # motion between 2 frames
            rot, trans, intrins, post_rot, post_trans = item['img_inputs'][1:]
            camera_meta_datas = {
                'rot': rot[None],
                'trans': trans[None],
                'intrins': intrins[None],
                'post_rot': post_rot[None],
                'post_trans': post_trans[None],
                'seq_ids': batch_seq_ids,
                'curr_to_prev_lidar': curr_to_prev_lidar[None],
                # 'timestep': torch.Tensor([(frame_id-slide_start)*5/(config.sample_interval)]),
                # 'frame_id': torch.Tensor([(frame_id-slide_start)*5]),
            }
            camera_meta_datas['timestep'] = torch.Tensor([(frame_id-slide_start)/10])
            for key in camera_meta_datas:
                if key not in ['seq_ids', 'curr_to_prev_lidar', 'timestep', 'frame_id']:
                    camera_meta_datas[key] = camera_meta_datas[key][:, 0:args.num_views]

            camera_meta_datas['size'] = (img_w, img_h) # (width, height)
        
            camera_meta_data_list.append(camera_meta_datas)       

        template = "This is a driving scene at %s (time %s). %s"
        prompt_list = []
        for fid in range(slide_start, slide_end):
            meta_data = sample[fid]['img_metas']
            if meta_data['location'] is None:
                prompt = meta_data['description']
            else:
                prompt = template%(meta_data['location'], meta_data['timeofday'], meta_data['description'])
            prompt_list.append(prompt)

        if args.object_condition:
            bbox_sequence = []
            curr_to_first = torch.eye(4)[None]
            for tid in range(slide_start, slide_end):
                if tid != slide_start:
                    cur_to_prev = camera_meta_data_list[tid-slide_start]['curr_to_prev_lidar']
                    curr_to_first = curr_to_first @ cur_to_prev
                frame_data = sample[tid]
                frame_boxes_3d = frame_data['gt_bboxes_3d']
                frame_labels = frame_data['gt_labels_3d']
                
                if len(frame_labels) > 0:
                    xyz = frame_boxes_3d.bottom_center
                    dims = frame_boxes_3d.dims
                    yaw = frame_boxes_3d.yaw
                    
                    xyz, yaw = convert_bbox(xyz, yaw, first_to_seq_lidar[:3,:3], first_to_seq_lidar[:3, 3])
                    boxes = torch.cat([xyz, dims, yaw[:, None]], dim=-1)
                    frame_boxes_3d = LiDARInstance3DBoxes(boxes)
                
                    frame_corners = frame_boxes_3d.corners.to(device, non_blocking=True) 
                    frame_labels = frame_labels.to(device, non_blocking=True) 
                    frame_bbox_num = len(frame_labels)
                    
                    
                    frame_corners = torch.cat([frame_corners], dim=0)
                    frame_labels = torch.cat([frame_labels], dim=0)
                else:
                    frame_corners = torch.zeros((0, 8, 3)).to(device, non_blocking=True)
                    frame_labels = torch.zeros(0).to(device, non_blocking=True)
                    frame_bbox_num = 0

                bbox_sequence.append([[frame_corners, frame_labels, frame_bbox_num]])
        else:
            bbox_sequence = None


        if map_condition:
            map_sequence = []
            curr_to_first = torch.eye(4)[None]
            for tid in range(slide_start, slide_end):
                if tid != slide_start:
                    cur_to_prev = camera_meta_data_list[tid-slide_start]['curr_to_prev_lidar']
                    curr_to_first = curr_to_first @ cur_to_prev
                    
                frame_data = sample[tid]
                frame_map_labels = frame_data['map_type_labels']
                frame_map_points = frame_data['map_sampled_points']

                if len(frame_map_labels) > 0:
                    frame_map_points = convert_points(frame_map_points, first_to_seq_lidar[:3,:3], first_to_seq_lidar[:3, 3])
                    frame_map_num = len(frame_map_labels)
                    
                    frame_map_points = frame_map_points.to(device, non_blocking=True)
                    frame_map_labels = frame_map_labels.to(device, non_blocking=True)
                else:
                    frame_map_points = torch.zeros((0, map_sample_points_num, 3)).to(device, non_blocking=True)
                    frame_map_labels = torch.zeros(0).to(device, non_blocking=True)
                    frame_map_num = 0
                
                map_sequence.append([[frame_map_points, frame_map_labels, frame_map_num]])
        else:
            map_sequence = None

        for camera_meta_datas in camera_meta_data_list:
            for key in camera_meta_datas:
                if key not in ['size']:
                    camera_meta_datas[key] = camera_meta_datas[key].to(device, non_blocking=True) 
        
        
        if slide_id == 0:
            if use_first_frame > 0:
                temporal_gt_leak = use_first_frame
                # Prepare input images (T=4 x num_views, 3, h, w)
                image_list = []
                for tid in range(use_first_frame):
                    image_list.append(sample[tid]['img_inputs'][0])

                start_imgs = torch.cat(image_list, dim=0).to(device)
                outputs = vae.encode(start_imgs, scale_schedule=scale_schedule)
                
                # Set the quant idx for each scale
                gt_ms_idx_Bmls = outputs[3]
                temporal_gt_ls_Bl = {}
                for tid in range(use_first_frame):
                    temporal_gt_ls_Bl[tid] = {}
                    for sid, scale_gt in enumerate(gt_ms_idx_Bmls):
                        scale_gt_t = scale_gt[tid*num_views:(tid+1)*num_views]
                        temporal_gt_ls_Bl[tid][sid] = scale_gt_t.to(torch.int64)
            else:
                temporal_gt_leak = 0
                temporal_gt_ls_Bl = {}
        else:
            temporal_gt_leak = timesteps - slides
            start_imgs = TV3HW[-temporal_gt_leak:].to(torch.float32)
            
            start_imgs = (start_imgs / 255 - 0.5) / 0.5
            
            start_imgs = start_imgs.flatten(0, 1) # [T*V, 3, H, W]
            outputs = vae.encode(start_imgs, scale_schedule=scale_schedule)

            gt_ms_idx_Bmls = outputs[3]
            temporal_gt_ls_Bl = {}
            for tid in range(temporal_gt_leak):
                temporal_gt_ls_Bl[tid] = {}
                for sid, scale_gt in enumerate(gt_ms_idx_Bmls):
                    scale_gt_t = scale_gt[tid*num_views:(tid+1)*num_views]
                    temporal_gt_ls_Bl[tid][sid] = scale_gt_t.to(torch.int64)
        
        generated_image = gen_one_frame_raynova(
            infinity,
            vae,
            text_tokenizer,
            text_encoder,
            prompt_list,
            camera_meta_data_list,
            g_seed=seed,
            gt_leak=0,
            gt_ls_Bl=None,
            temporal_gt_leak=temporal_gt_leak,
            temporal_gt_ls_Bl=temporal_gt_ls_Bl,
            cfg_list=cfg,
            tau_list=tau,
            scale_schedule=scale_schedule,
            cfg_insertion_layer=[args.cfg_insertion_layer],
            vae_type=args.vae_type,
            sampling_per_bits=args.sampling_per_bits,
            enable_positive_prompt=enable_positive_prompt,
            bbox_sequence=bbox_sequence,
            map_sequence=map_sequence,
        )

        TV3HW = generated_image.permute(1,0,4,2,3) # [T, V, 3, H, W]
        if slide_id == 0:
            video_imgs.extend(torch.unbind(TV3HW, dim=0))
        else:
            video_imgs.extend(torch.unbind(TV3HW[-slides:], dim=0))

    return video_imgs

In [5]:
def visualize_box(sample, video_imgs):
    all_bbox_imgs = []
    curr_to_first_lidar = torch.eye(4)
    curr_to_first_lidar_list = [curr_to_first_lidar]
    for item in sample[1:]:
        meta_data = item['img_metas']
        curr_to_first_lidar = curr_to_first_lidar @ meta_data['curr_to_prev_lidar_rt']
        curr_to_first_lidar_list.append(curr_to_first_lidar)
    
    camera_meta_data_list = []
    for frame_id in range(len(sample)):
        item = sample[frame_id]
        meta_data = item['img_metas']
        batch_seq_ids = torch.zeros(1).int()  # No need to check sequence validation
        curr_to_prev_lidar = meta_data['curr_to_prev_lidar_rt']
        rot, trans, intrins, post_rot, post_trans = item['img_inputs'][1:]
        # camera_meta_datas['size'] = (img_w, img_h) # (width, height)
        
        camera2lidar = torch.stack([torch.eye(4)]*rot.shape[0],dim=0)
        camera2lidar[:, :3, :3] = rot
        camera2lidar[:, :3, 3] = trans
        lidar2camera = torch.inverse(camera2lidar)
        
        camera2image = torch.stack([torch.eye(4)]*rot.shape[0],dim=0)
        camera2image[:, :3, :3] = intrins
        
        lidar2image = camera2image @ lidar2camera

        img_aug_matrix = torch.stack([torch.eye(4)]*rot.shape[0],dim=0)
        img_aug_matrix[:, :3, :3] = post_rot
        img_aug_matrix[:, :3, 3] = post_trans

        camera_meta_datas = {
            'rot': rot,
            'trans': trans,
            'intrins': intrins,
            'post_rot': post_rot,
            'post_trans': post_trans,
            'seq_ids': batch_seq_ids,
            'curr_to_prev_lidar': curr_to_prev_lidar,
            'img_aug_matrix': img_aug_matrix,
            'lidar2image': lidar2image,
        }
        if args.num_views == 1:
            for key in camera_meta_datas:
                if key not in ['seq_ids', 'curr_to_prev_lidar']:
                    camera_meta_datas[key] = camera_meta_datas[key][:, 0:1]


        camera_meta_data_list.append(camera_meta_datas)       

    bbox_sequence = []
    for tid in range(len(sample)):
        frame_data = sample[tid]
        gt_bboxes_3d = frame_data['gt_bboxes_3d']
        gt_labels_3d = frame_data['gt_labels_3d']

        if len(gt_labels_3d) > 0:
            xyz = gt_bboxes_3d.bottom_center
            dims = gt_bboxes_3d.dims
            yaw = gt_bboxes_3d.yaw
            first_to_seq_lidar = torch.inverse(curr_to_first_lidar_list[tid])
            
            xyz, yaw = convert_bbox(xyz, yaw, first_to_seq_lidar[:3,:3], first_to_seq_lidar[:3, 3])
            boxes = torch.cat([xyz, dims, yaw[:, None]], dim=-1)
            gt_bboxes_3d = LiDARInstance3DBoxes(boxes)
            bbox_num = boxes.shape[0]
        else:
            gt_bboxes_3d = LiDARInstance3DBoxes(torch.zeros((0, 7)))
            gt_labels_3d = torch.zeros(0)
            bbox_num = 0
        
        gt_labels_3d = gt_labels_3d.to(torch.int32)
        bbox_sequence.append([gt_bboxes_3d, gt_labels_3d, bbox_num])

    for tid in range(len(sample)):
        frame_meta_data = camera_meta_data_list[tid]
        frame_bbox = bbox_sequence[tid]
        
        frame_images = video_imgs[tid].permute(0, 2, 3, 1)
        frame_images = torch.unbind(frame_images, dim=0)
        frame_images = [Image.fromarray(img.cpu().numpy()) for img in frame_images]

        frame_data = {
            'gt_bboxes_3d': bbox_sequence[tid][0],
            'gt_labels_3d': bbox_sequence[tid][1],
            'lidar2image': camera_meta_data_list[tid]['lidar2image'],
            'img_aug_matrix': camera_meta_data_list[tid]['img_aug_matrix'],
        }
        
        bbox_imgs = draw_box_on_imgs(frame_data, frame_images, np.array(class_names))
        
        bbox_imgs = [torch.from_numpy(np.array(img)) for img in bbox_imgs]
        bbox_imgs = torch.stack(bbox_imgs, dim=0)
        bbox_imgs = bbox_imgs.permute(0, 3, 1, 2)
        all_bbox_imgs.append(bbox_imgs)
    
    return all_bbox_imgs

In [6]:
def visualize_map_points(sample, video_imgs):
    all_map_imgs = []
    curr_to_first_lidar = torch.eye(4)
    curr_to_first_lidar_list = [curr_to_first_lidar]
    for item in sample[1:]:
        meta_data = item['img_metas']
        curr_to_first_lidar = curr_to_first_lidar @ meta_data['curr_to_prev_lidar_rt']
        curr_to_first_lidar_list.append(curr_to_first_lidar)
    
    camera_meta_data_list = []
    for frame_id in range(len(sample)):
        item = sample[frame_id]
        meta_data = item['img_metas']
        batch_seq_ids = torch.zeros(1).int()  # No need to check sequence validation
        curr_to_prev_lidar = meta_data['curr_to_prev_lidar_rt']
        rot, trans, intrins, post_rot, post_trans = item['img_inputs'][1:]
        # camera_meta_datas['size'] = (img_w, img_h) # (width, height)
        
        camera2lidar = torch.stack([torch.eye(4)]*rot.shape[0],dim=0)
        camera2lidar[:, :3, :3] = rot
        camera2lidar[:, :3, 3] = trans
        lidar2camera = torch.inverse(camera2lidar)
        
        camera2image = torch.stack([torch.eye(4)]*rot.shape[0],dim=0)
        camera2image[:, :3, :3] = intrins
        
        lidar2image = camera2image @ lidar2camera

        img_aug_matrix = torch.stack([torch.eye(4)]*rot.shape[0],dim=0)
        img_aug_matrix[:, :3, :3] = post_rot
        img_aug_matrix[:, :3, 3] = post_trans

        camera_meta_datas = {
            'rot': rot,
            'trans': trans,
            'intrins': intrins,
            'post_rot': post_rot,
            'post_trans': post_trans,
            'seq_ids': batch_seq_ids,
            'curr_to_prev_lidar': curr_to_prev_lidar,
            'img_aug_matrix': img_aug_matrix,
            'lidar2image': lidar2image,
        }

        camera_meta_data_list.append(camera_meta_datas)       

    for tid in range(len(sample)):
        frame_data = sample[tid]
        map_points = frame_data['map_sampled_points']
        map_labels = frame_data['map_type_labels']
        
        if len(map_labels) > 0:
            first_to_seq_lidar = torch.inverse(curr_to_first_lidar_list[tid])
            
            frame_points = convert_points(map_points, first_to_seq_lidar[:3,:3], first_to_seq_lidar[:3, 3])
        else:
            frame_points = None
                
        frame_images = video_imgs[tid].permute(0, 2, 3, 1)
        frame_images = torch.unbind(frame_images, dim=0)
        frame_images = [Image.fromarray(img.cpu().numpy()) for img in frame_images]

        frame_data = {
            'map_type_labels': map_labels,
            'map_sampled_points': frame_points,
            'lidar2image': camera_meta_data_list[tid]['lidar2image'],
            'img_aug_matrix': camera_meta_data_list[tid]['img_aug_matrix'],
        }
        
        map_imgs = draw_map_points_on_imgs(frame_data, frame_images, np.array(map_names))
        
        map_imgs = [torch.from_numpy(np.array(img)) for img in map_imgs]
        map_imgs = torch.stack(map_imgs, dim=0)
        map_imgs = map_imgs.permute(0, 3, 1, 2)
        all_map_imgs.append(map_imgs)
    
    return all_map_imgs

In [7]:
def save_imgs(images, suffix=''):
    if save_format == 'gif':
        TV3HW = torch.stack(images, dim=0)
        TV3HW = TV3HW.flatten(0, 1)
        vthw = torchvision.utils.make_grid(TV3HW, nrow=args.num_views, padding=0, pad_value=1.0)
        vthw = vthw.clone().permute(1, 2, 0).cpu().numpy()
        vthw = PImage.fromarray(vthw.astype(np.uint8))
        vthw.save(f"./demo_output/output_video_{suffix}.png")

        gif_frames = []
        for image in images:
            frame = torch.unbind(image, dim=0)
            frame = torch.cat(frame, dim=2)
            frame = frame.permute(1, 2, 0).cpu().numpy()
            gif_frames.append(Image.fromarray(frame))
        gif_frames[0].save(f"./demo_output/output_video_{suffix}.gif", save_all=True, append_images=gif_frames[1:], duration=500, loop=0)

    else:
        assert save_format == 'mp4'
        # Initialize video writer
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Use 'mp4v' for MP4 format
        video = cv2.VideoWriter(f"./demo_output/output_video_{suffix}.mp4", fourcc, 10, (img_w*num_views, img_h))

        # Add images to the video
        for image in images:
            frame = torch.unbind(image, dim=0)
            frame = torch.cat(frame, dim=2)
            frame = frame.permute(1, 2, 0).cpu().numpy()
            frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
            video.write(frame)

        # Release the video writer
        video.release()

In [8]:
def get_gt_images(sample):
    gt_images = []
    for tid in range(len(sample)):
        frame_sample = sample[tid]
        frame_images = frame_sample['img_inputs'][0].clone()

        frame_images = (frame_images + 1) / 2
        frame_images = (frame_images*255).to(torch.uint8)
        
        gt_images.append(frame_images)
    return gt_images

In [ ]:
import pickle

# meta data:
# curr_to_first_lidar_rt [4, 4]: transform from current frame to the first frame
# curr_to_prev_lidar_rt [4, 4]: transform from current frame to the previous frame
# location: location of the scene
# timeofday: time of day
# description: text description of the scene for each frame

# camera paramaters (in a list orderly): [rot, trans, intrins, post_rot, post_trans]
# rot / trans: camera extrinsic parameter (rotation [V, 3, 3] and translation [V, 3]) for each the V views
# intrins: camera intrinsic parameter for each view [V, 3, 3]
# post_rot: camera rotation / scaling after image augmentation (our default setting only includes scaling) [V, 3, 3]
# post_trans: camera translation after image augmentation (our default setting is 0) [V, 3]


# optional conditions: 
# If you don't use the conditions, please maintain the torch Tensor structure and set B=0 or N=0
# object:
# gt_bboxes_3d: ground truth bounding boxes for each frame [B, 7] (B bboxes of 7D, refer to nuscenes_tools.nuscenes_utils.box3d_instance.LiDARInstance3DBoxes for details)
# gt_labels_3d: labels for each bounding box [B,] ('car', 'truck', 'construction_vehicle', 'bus', 'trailer', 'traffic_barrier', 'motorcycle', 'bicycle', 'pedestrian', 'traffic_cone', 'vehicle', 'cyclist', 'debris', 'traffic_light', 'unknown',)
# map:
# map_sampled_points: sampled map points for each frame [N, P, 3]: N map elements, P  3D points are randomly sampled on each elements 
# map_type_labels: type labels for each map point [N,]: ['road_line_solid_single_white', 'road_line_broken_single_white', 'lane_surface_street', 'road_edge_sidewalk', 'road_edge_boundary', 'lane_surface_unstructure', 'crosswalk', 'road_line_solid_single_yellow']

# input images: (T frames, V views)
# only the first "use_first_frame" frames are used as the initial frames. Others are used only for visualization. You can set all other frames as placeholder images.

meta_keys = ['curr_to_first_lidar_rt', 'curr_to_prev_lidar_rt', 'location', 'timeofday', 'description']


def load_sample_from_file(root_dir):
    
    sample = []
    for fid in range(num_frames):
        with open(f"{root_dir}/frame_{fid}/item_dict.pkl", "rb") as f:
            file_dict = pickle.load(f)
            
        frame_dict = {}
        frame_dict['img_metas'] = {}
        for key in meta_keys:
            frame_dict['img_metas'][key] = file_dict['img_metas'][key]
            
        frame_dict['gt_bboxes_3d'] = LiDARInstance3DBoxes(file_dict['gt_bboxes_3d'])
        frame_dict['gt_labels_3d'] = file_dict['gt_labels_3d']
        frame_dict['map_sampled_points'] = file_dict['map_sampled_points'].to(torch.float32)
        frame_dict['map_type_labels'] = file_dict['map_type_labels']
                                
        
        mv_imgs = []
        for vid in range(num_views):
            img = Image.open(f"{root_dir}/frame_{fid}/frame_images_v{vid}.png")
            mv_imgs.append(np.array(img))
            
        mv_imgs = np.stack(mv_imgs, axis=0)
        mv_imgs = mv_imgs / 255.0 * 2 - 1
        mv_imgs = torch.Tensor(mv_imgs).permute(0, 3, 1, 2)
        
        frame_dict['img_inputs'] = [mv_imgs] + list(file_dict['camera_params'])
                    
        sample.append(frame_dict)
    return sample


In [10]:
from PIL import Image

sample = load_sample_from_file("demo_sample")
gt_images = get_gt_images(sample)
video_imgs = generation(sample)
save_imgs(video_imgs)

if object_condition:
    gt_images = visualize_box(sample, gt_images)
    video_imgs = visualize_box(sample, video_imgs)
    save_imgs(gt_images, 'gt_bbox')
    save_imgs(video_imgs, 'bbox')

if map_condition:
    gt_images = visualize_map_points(sample, gt_images)
    video_imgs = visualize_map_points(sample, video_imgs)
    save_imgs(gt_images, 'gt_map')
    save_imgs(video_imgs, 'map')
    
    

Slide id: 0 7
prompt=This is a driving scene at us-nv-las-vegas-strip (time 16:48). The scene unfolds on a bright, cloudless afternoon along a wide, six-lane urban boulevard resembling the Las Vegas Strip. Directly ahead, a queue of sedans, SUVs and a white minivan grinds forward beneath an overhead pedestrian bridge plastered with green exit signs for Route 15 and Tropicana Avenue. To the left, a silver luxury sedan inches past palm trees and concrete barriers, while to the right, a dark compact SUV hugs the curb beside a low sidewalk guardrail. All lanes are congested with vehicles, and no pedestrians are visible in the crosswalks. Sharp shadows from nearby buildings and billboard advertising suggest it’s midday, and the asphalt appears clean and dry.
prompt=This is a driving scene at us-nv-las-vegas-strip (time 16:48). The scene unfolds on a bright, cloudless afternoon along a wide, six-lane urban boulevard resembling the Las Vegas Strip. Directly ahead, a queue of sedans, SUVs and 

cost: 19.488136053085327, infinity cost=19.330891132354736
Slide id: 1 7
prompt=This is a driving scene at us-nv-las-vegas-strip (time 16:48). The scene unfolds on a bright, cloudless afternoon along a wide, six-lane urban boulevard resembling the Las Vegas Strip. Directly ahead, a queue of sedans, SUVs and a white minivan grinds forward beneath an overhead pedestrian bridge plastered with green exit signs for Route 15 and Tropicana Avenue. To the left, a silver luxury sedan inches past palm trees and concrete barriers, while to the right, a dark compact SUV hugs the curb beside a low sidewalk guardrail. All lanes are congested with vehicles, and no pedestrians are visible in the crosswalks. Sharp shadows from nearby buildings and billboard advertising suggest it’s midday, and the asphalt appears clean and dry.
prompt=This is a driving scene at us-nv-las-vegas-strip (time 16:48). The scene unfolds on a bright, cloudless afternoon along a wide, six-lane urban boulevard resembling the La